In [81]:
import pandas as pd
import numpy as np
import pyarrow.dataset as ds
import s3fs

## Dataset Import

In [82]:
# Import FIP Dataset

s3_path_fip = (
    "s3://m3-intel-hub-dp-us-east-1-517292-prod/"
    "publish/data-product/financial_inventory_projection_report_network_update/"
)

dataset = ds.dataset(
    s3_path_fip,
    format="parquet",
    partitioning="hive" 
)

table = dataset.to_table(
    filter=(
        ds.field("date").isin(["202612"])  # "202712", "202812" 
     ) 
        # & 
    # (
    #     ds.field("corporate_brand").isin(['REVLIMID'])#,"All Other Pharmaceut"'ABRAXANE',])
    # ) &
    # (
    #     ds.field("snapshot_date") >= "2025-10-01"
    # ) 
    & ~(
        (ds.field("snapshot_date") == "2026-01-23") &
        (ds.field("snapshot_type") == "friday")
    )
)


df_fip = table.to_pandas()
df_fip.head()

,material,plant,date,quantity,total_cost,concost_source,unit_of_measure,cost_per_unit,source,corporate_brand,material_type,development_lifecycle_status,enterprise_category,enterprise_sub_category,dosage_form_parent,corp_brand_id,network_or_business_unit,snapshot_type,snapshot_date
0,1371110,1734,202612,2460.0,NaN,missing,None,NaN,rr,nan,HALB,CLINICAL,OTHER PROCESS MATERIALS,OTHER PROCESS MATERIALS,nan,00000nan,None,friday,2025-07-18
1,1379260,1734,202612,77.9,NaN,missing,None,NaN,rr,nan,HALB,CLINICAL,OTHER PROCESS MATERIALS,OTHER PROCESS MATERIALS,nan,00000nan,None,friday,2025-07-18
2,1387184,1734,202612,3.5,NaN,missing,None,NaN,rr,nan,HALB,CLINICAL,OTHER PROCESS MATERIALS,OTHER PROCESS MATERIALS,nan,00000nan,None,friday,2025-07-18
3,1457614,2061,202612,37001.0,NaN,missing,None,NaN,rr,All Other Pharmaceut,UNBW,nan,nan,nan,nan,00201790,PHARMA,friday,2025-07-18
4,1457656,2061,202612,100.0,NaN,missing,None,NaN,rr,All Other Pharmaceut,UNBW,nan,nan,nan,nan,00201790,PHARMA,friday,2025-07-18


In [83]:
df_fip.shape

(933606, 19)

In [84]:
# Import Plant Type Data

s3_path_plants = (
    "s3://m3-intel-hub-dp-us-east-1-517292-prod/"
    "refined/data-asset/fin_inv_proj/"
    "bms_internal_vs_external_plants/"
    "bms_internal_vs_external_plants.parquet"
)

df_plants = pd.read_parquet(s3_path_plants)
#df_plants.head()

In [85]:
# Import node type Dataset

fs = s3fs.S3FileSystem()  # uses SageMaker execution role

parquet_files_nt = fs.glob(
    "s3://m3-intel-hub-dp-us-east-1-517292-prod/"
    "dbt_intelligence_hub/intelligence_hub_db_sandbox_staging/"
    "src__sap_t001w/data/*.parquet"
)

df_ntype = pd.read_parquet(
    parquet_files_nt,
    engine="pyarrow",
    dtype_backend="pyarrow",
    filesystem=fs
)

#df_ntype.head()

In [86]:
# Import Material Master Data

fs = s3fs.S3FileSystem()  # uses SageMaker execution role

parquet_files = fs.glob(
    "s3://m3-intel-hub-dp-us-east-1-517292-prod/"
    "refined/data-asset/fin_inv_proj/"
    "sap_material_master/data/*.parquet"
)

df_mm = pd.read_parquet(
    parquet_files,
    engine="pyarrow",
    dtype_backend="pyarrow",
    filesystem=fs
)

#df_mm.head()

In [87]:
# import boto3

# s3 = boto3.client("s3")

# bucket = "m3-intel-hub-dp-us-east-1-517292-prod"
# prefix = "dbt_intelligence_hub/intelligence_hub_db_sandbox_staging/src__sap_t001w"

# response = s3.list_objects_v2(
#     Bucket=bucket,
#     Prefix=prefix
# )

# if "Contents" in response:
#     for obj in response["Contents"]:
#         print(obj["Key"], obj["Size"])
# else:
#     print("No objects found or no access.")

## Data Prep

In [88]:
# Create has_non_zero flag at material–plant level

df_fip["has_non_zero"] = (
    df_fip
    .groupby(["material", "plant"])["total_cost"]
    .transform(lambda x: (x != 0).any())
    .astype(int)
)

# Apply the filter
df_fip = df_fip.loc[df_fip["has_non_zero"] == 1].drop(columns="has_non_zero")

In [89]:
df_fip.shape

(682694, 19)

In [90]:
base1 = df_fip.copy()

In [91]:
# Join material master
mm_cols = [
    "material_number",
    "plant",
    "profit_center",
    #"corporate_brand",
    "brand_name",
    #"corp_brand_id",
    "material_description",
    #"material_type",
    "material_group",
    "old_material_number",
    "base_unit_of_measure",
    "unit_of_weight",
    #"development_lifecycle_status",
    "plant_specific_material_status",
    "mrp_type",
    "procurement_type",
    "safety_stock",
    "minimum_lot_size",
    "maximum_lot_size",
    "fixed_lot_size",
    "total_replenishment_lead_time",
    "total_shelf_life",
    "batch_management",
    "abc_indicator",
    #"valuation_class",
    #"price_unit",
    #"price_control_indicator",
]

material_master_sel = (
    df_mm[mm_cols]
    .drop_duplicates(subset=["material_number", "plant"])
)


base1 = base1.merge(
    material_master_sel,
    left_on=["material", "plant"],
    right_on=["material_number", "plant"],
    how="left",
    validate="m:1" 
)
base1 = base1.drop(columns=["material_number"])

base1 = base1.merge(
    df_plants[["Plant", "Plant Type"]],
    left_on=["plant"],
    right_on=["Plant"],
    how="left"
).drop(columns=["Plant"])

node_type_lkp = (
    df_ntype[["werks", "nodetype"]]
    .drop_duplicates(subset=["werks"])
)

base1 = base1.merge(
    node_type_lkp,
    left_on="plant",
    right_on="werks",
    how="left",
    validate="m:1"
).drop(columns=["werks"])

base1.shape

(682694, 39)

In [92]:
# rearrange columns for better readability
new_cols = [
    'corporate_brand',
    'material_type', 'development_lifecycle_status', 'enterprise_category',
    'enterprise_sub_category', 'dosage_form_parent', 'corp_brand_id',
    'network_or_business_unit',
    'profit_center', 'brand_name', 'material_description', 'material_group',
    'old_material_number', 'base_unit_of_measure', 'unit_of_weight',
    'plant_specific_material_status', 'mrp_type', 'procurement_type',
    'safety_stock', 'minimum_lot_size', 'maximum_lot_size',
    'fixed_lot_size', 'total_replenishment_lead_time', 'total_shelf_life',
    'batch_management', 'abc_indicator',
    'source', 'concost_source', 'Plant Type', 'nodetype',
    'unit_of_measure',
    'material', 'plant', 'date', 'quantity', 'total_cost',
    'cost_per_unit', 'snapshot_type', 'snapshot_date'
]

base1 = base1[new_cols]


In [93]:
base1.columns

Index(['corporate_brand', 'material_type', 'development_lifecycle_status',
       'enterprise_category', 'enterprise_sub_category', 'dosage_form_parent',
       'corp_brand_id', 'network_or_business_unit', 'profit_center',
       'brand_name', 'material_description', 'material_group',
       'old_material_number', 'base_unit_of_measure', 'unit_of_weight',
       'plant_specific_material_status', 'mrp_type', 'procurement_type',
       'safety_stock', 'minimum_lot_size', 'maximum_lot_size',
       'fixed_lot_size', 'total_replenishment_lead_time', 'total_shelf_life',
       'batch_management', 'abc_indicator', 'source', 'concost_source',
       'Plant Type', 'nodetype', 'unit_of_measure', 'material', 'plant',
       'date', 'quantity', 'total_cost', 'cost_per_unit', 'snapshot_type',
       'snapshot_date'],
      dtype='object')

In [110]:
# Add material entry flag 

base = base1.copy()
first_seen = (
    base.groupby(["material", "plant","date"])["snapshot_date"]
      .transform("min")
)

base["sku_status"] = np.where(
    base["snapshot_date"] == first_seen,
    "NEW",
    "EXISTING"
)


In [111]:
# fip copy df for data prep
df = base.copy()
df["snapshot_date"] = pd.to_datetime(df["snapshot_date"])


# snapshot lookup table
snapshot_calendar = (
    df[["snapshot_type", "snapshot_date"]]
    .drop_duplicates()
    .sort_values(["snapshot_type", "snapshot_date"])
)


# attach prev snapshot to snapshot calendar
snapshot_calendar["prev_snapshot_date"] = (
    snapshot_calendar
    .groupby("snapshot_type")["snapshot_date"]
    .shift(1)
)
snapshot_calendar      # comparing bd13 - bd13 snapshots and friday-friday snapshots. no bd13-fri snapshots

,snapshot_type,snapshot_date,prev_snapshot_date
121767,bd13,2025-08-26,NaT
203326,bd13,2025-09-18,2025-08-26
291085,bd13,2025-10-17,2025-09-18
393323,bd13,2025-11-19,2025-10-17
497156,bd13,2025-12-17,2025-11-19
621780,bd13,2026-01-23,2025-12-17
0,friday,2025-07-18,NaT
20237,friday,2025-07-25,2025-07-18
40477,friday,2025-08-01,2025-07-25
60737,friday,2025-08-08,2025-08-01


In [112]:
# Attach previous snapshot date to each row

df = df.merge(
    snapshot_calendar[["snapshot_type", "snapshot_date", "prev_snapshot_date"]],
    on=["snapshot_type", "snapshot_date"],
    how="left"
)

In [130]:
# Prepare current and previous frames

# Current snapshot frame
current_df = df.copy()

current_df = current_df.rename(columns={
    "quantity": "quantity_curr",
    "cost_per_unit": "cost_per_unit_curr",
    "total_cost": "total_cost_curr",
})

# Previous snapshot frame
previous_df = df.rename(columns={
    "snapshot_date": "snapshot_date_prev",
    "quantity": "quantity_prev",
    "cost_per_unit": "cost_per_unit_prev",
    "total_cost": "total_cost_prev",
})[
    [
        "material",
        "plant",
        "date",
        "snapshot_type",
        "snapshot_date_prev",
        "quantity_prev",
        "cost_per_unit_prev",
        "total_cost_prev",
    ]
]


# Join current to previous snapshot
rca_base = current_df.merge(
    previous_df,
    left_on=[
        "material",
        "plant",
        "date",
        "snapshot_type",
        "prev_snapshot_date",
    ],
    right_on=[
        "material",
        "plant",
        "date",
        "snapshot_type",
        "snapshot_date_prev",
    ],
    how="left"
)

In [131]:
rca_base.shape

(683718, 45)

In [132]:
# Flags for material–plant presence 
# 1. Identify PHASED IN in current snapshot
# Present now, but not present in previous snapshot
rca_base["phased_in"] = (
    rca_base["prev_snapshot_date"].notna() &
    rca_base["quantity_prev"].isna()
)


# 2. Identify PHASED OUT in current snapshot
# Present in previous snapshot but missing in current snapshot

# Identify valid previous snapshots (calendar-safe)
valid_prev_snapshots = (
    snapshot_calendar["prev_snapshot_date"]
        .dropna()
        .unique()
)

# Restrict previous snapshot data to valid transitions
previous_df_valid = previous_df[
    previous_df["snapshot_date_prev"].isin(valid_prev_snapshots)
]

# Anti-join: rows present in previous but missing in current
prev_only = previous_df_valid.merge(
    current_df[
        [
            "material",
            "plant",
            "date",
            "snapshot_type",
            "prev_snapshot_date",
        ]
    ],
    left_on=[
        "material",
        "plant",
        "date",
        "snapshot_type",
        "snapshot_date_prev",
    ],
    right_on=[
        "material",
        "plant",
        "date",
        "snapshot_type",
        "prev_snapshot_date",
    ],
    how="left",
    indicator=True
).query("_merge == 'left_only'")

# Mark phased out rows
prev_only["phased_out"] = True

# Ensure column alignment for concat
for col in rca_base.columns:
    if col not in prev_only.columns:
        prev_only[col] = np.nan

# Current snapshot rows are NOT phased out
rca_base["phased_out"] = False


# 3. Combine current + phased out rows
final_rca_frame = pd.concat(
    [rca_base, prev_only[rca_base.columns]],
    ignore_index=True
)

# 4. Normalize boolean flags
for col in [
    "phased_in",
    "phased_out",
]:
    final_rca_frame[col] = (
        final_rca_frame[col]
            .replace({1: True, 0: False})
            .fillna(False)
            .astype("boolean")
    )

/tmp/ipykernel_422638/799287440.py:67: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_rca_frame = pd.concat(
/tmp/ipykernel_422638/799287440.py:80: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)


In [133]:
# Cost Zero to Available flagging

prev_cost = final_rca_frame["cost_per_unit_prev"]
curr_cost = final_rca_frame["cost_per_unit_curr"]

# Treat nulls as zero
prev_cost = prev_cost.fillna(0)
curr_cost = curr_cost.fillna(0)

# Treat very small values as zero
prev_is_zero = prev_cost.abs() < 1e-2
curr_is_zero = curr_cost.abs() < 1e-2

conditions = [
    prev_is_zero & (~curr_is_zero),   # 0 → positive
    (~prev_is_zero) & curr_is_zero    # positive → 0
]

choices = [
    "COST_INTRODUCED",
    "COST_REMOVED"
]

final_rca_frame["cost_transition_flag"] = np.select(
    conditions,
    choices,
    default="NO_CHANGE"
)


In [134]:
final_rca_frame.shape

(703000, 48)

In [135]:
snapshot_mapping_check = (
    final_rca_frame
    .loc[:, ["snapshot_type", "snapshot_date", "prev_snapshot_date"]]
    .drop_duplicates()
    .sort_values(["snapshot_type", "snapshot_date"])
)

snapshot_mapping_check

,snapshot_type,snapshot_date,prev_snapshot_date
122127,bd13,2025-08-26,NaT
203898,bd13,2025-09-18,2025-08-26
291829,bd13,2025-10-17,2025-09-18
394312,bd13,2025-11-19,2025-10-17
498180,bd13,2025-12-17,2025-11-19
622804,bd13,2026-01-23,2025-12-17
683956,bd13,NaT,NaT
0,friday,2025-07-18,NaT
20237,friday,2025-07-25,2025-07-18
40549,friday,2025-08-01,2025-07-25


Error handling

1. Division by zero & invalid math -
    Previous quantity = 0
    Previous cost = 0
    Previous FIP = 0
    Volatility = 0
2. Min rolling window
   less than min periods of 3
   Newly introduced SKU
   Flat history causing volatility = 0
3. double counting due to duplicates
4. missing prev snapshot data - nulls
   SKU appears for first time
   SKU disappears and reappears
   Previous quantity / cost not available
5. Extreme values in the history
    Very large quantities or costs in the past pushing the present numbers. The outliers in the past affects the current numbers. Exlcude the historical outliers
   

# Data Transformations

### Driver Calculations

In [136]:
# Step 1: Compute raw change metrics (always runs)

# Base deltas

final_rca_frame["delta_quantity"] = (
    final_rca_frame["quantity_curr"] - final_rca_frame["quantity_prev"]
)

final_rca_frame["delta_cost_per_unit"] = (
    final_rca_frame["cost_per_unit_curr"] - final_rca_frame["cost_per_unit_prev"]
)


# Impact decomposition

# Quantity Impact 
final_rca_frame["quantity_impact"] = (
    final_rca_frame["delta_quantity"] *
    final_rca_frame["cost_per_unit_prev"]
)

# Cost Impact
final_rca_frame["cost_impact"] = (
    final_rca_frame["delta_cost_per_unit"] * final_rca_frame["quantity_prev"]
)

# Intercation
final_rca_frame["interaction_impact"] = (
    final_rca_frame["delta_quantity"] *
    final_rca_frame["delta_cost_per_unit"]
)

# Total Change
final_rca_frame["total_fip_change"] = (
    final_rca_frame["quantity_impact"] +
    final_rca_frame["cost_impact"] +
    final_rca_frame["interaction_impact"]
)

# Core Metrics

# delta_quantity_pct 
final_rca_frame["delta_quantity_pct"] = np.where(
    final_rca_frame["quantity_prev"] > 0,
    final_rca_frame["delta_quantity"] / final_rca_frame["quantity_prev"],
    np.nan
)

# delta_cost_per_unit_pct

final_rca_frame["delta_cost_per_unit_pct"] = np.where(
    final_rca_frame["cost_per_unit_prev"] > 0,
    final_rca_frame["delta_cost_per_unit"] / final_rca_frame["cost_per_unit_prev"],
    np.nan
)


# Contribution shares (absolute, normalized)
impact_abs_sum_qc = (
    final_rca_frame["quantity_impact"].abs() +
    final_rca_frame["cost_impact"].abs()
)

impact_abs_sum_all = (
    impact_abs_sum_qc +
    final_rca_frame["interaction_impact"].abs()
)


# quantity_impact_pct_of_total 
final_rca_frame["quantity_impact_pct_of_total"] = np.where(
    impact_abs_sum_qc > 0,
    final_rca_frame["quantity_impact"].abs() / impact_abs_sum_qc,
    0
)

# cost_impact_pct_of_total 
final_rca_frame["cost_impact_pct_of_total"] = np.where(
    impact_abs_sum_qc > 0,
    final_rca_frame["cost_impact"].abs() / impact_abs_sum_qc,
    0
)

# interaction_pct
final_rca_frame["interaction_pct"] = np.where(
    impact_abs_sum_all > 0,
    final_rca_frame["interaction_impact"].abs() / impact_abs_sum_all,
    0
)


# Dominance Score

final_rca_frame["abs_qty_impact"] = final_rca_frame["quantity_impact"].abs()
final_rca_frame["abs_cost_impact"] = final_rca_frame["cost_impact"].abs()

final_rca_frame["dominance_score"] = np.where(
    (final_rca_frame["abs_qty_impact"] + final_rca_frame["abs_cost_impact"]) == 0,
    0.0,  # avoid divide-by-zero → treat as neutral
    (final_rca_frame["abs_qty_impact"] - final_rca_frame["abs_cost_impact"]) /
    (final_rca_frame["abs_qty_impact"] + final_rca_frame["abs_cost_impact"])
)

### Data Sufficiency Metrics

In [137]:
final_rca_frame = final_rca_frame.sort_values(
    ["material", "plant","date", "snapshot_date"]
)

final_rca_frame["is_qty_present_greater0"] = (
    final_rca_frame["quantity_curr"] > 0
).astype(int)

# 1) history_weeks_count
final_rca_frame["history_weeks_count_greater0"] = (
    final_rca_frame
    .groupby(["material", "plant","date"])["is_qty_present_greater0"]
    .rolling(window=12, min_periods=1)
    .sum()
    .reset_index(level=[0,1,2], drop=True)
)

final_rca_frame["hist_count"] = (
    final_rca_frame
        .groupby(["material", "plant", "date"])["quantity_curr"]
        .transform(lambda x: x.notna().cumsum() - 1)
)

final_rca_frame["has_sufficient_6periods"] = (
    final_rca_frame["hist_count"] >= 6
)


# 2) presence_stability_score
final_rca_frame["presence_stability_score"] = (
    final_rca_frame["hist_count"] / 12
).clip(upper=1.0)

# 3) History confidence (intentionally mirrors stability)
final_rca_frame["history_confidence"] = (
    final_rca_frame["hist_count"] / 12
).clip(upper=1.0)


# 4) sku_presence_class
conditions = [
    final_rca_frame["phased_in"] == True,  
    final_rca_frame["phased_out"] == True,
    (
        (final_rca_frame["quantity_prev"] == 0) &
        (final_rca_frame["quantity_curr"] > 0) &
        (final_rca_frame["history_weeks_count_greater0"] < 6)
    ),
    (
        (final_rca_frame["presence_stability_score"] >= 0.75) &
        (final_rca_frame["history_weeks_count_greater0"] >= 6)
    )
]
 
choices = [
    "ENTERED",
    "OUT",
    "REACTIVATED",
    "STABLE"
]
 
final_rca_frame["sku_presence_class"] = np.select(
    conditions,
    choices,
    default="ERRATIC"
)


# 5) uom_change_flag
final_rca_frame = final_rca_frame.sort_values(
    ["material", "plant","date", "snapshot_date"]
)

final_rca_frame["unit_of_measure_prev"] = (
    final_rca_frame
    .groupby(["material", "plant","date"])["base_unit_of_measure"]
    .shift(1)
)

final_rca_frame["uom_change_flag"] = (
    final_rca_frame["unit_of_measure_prev"].notna() &
    (final_rca_frame["unit_of_measure_prev"] != final_rca_frame["unit_of_measure"])
)


# 6) duplicate_sku_plant_flag
dup_counts = (
    final_rca_frame
    .groupby(["material", "plant","date", "snapshot_date"])
    .size()
    .rename("dup_count")
    .reset_index()
)
final_rca_frame = final_rca_frame.merge(
    dup_counts,
    on=["material", "plant","date", "snapshot_date"],
    how="left"
)
final_rca_frame["duplicate_sku_plant_flag"] = (
    final_rca_frame["dup_count"] > 1
)


# 7) dominance_instability_flag

final_rca_frame["dominance_score_var_4"] = (
    final_rca_frame
    .groupby(["material", "plant","date"])["dominance_score"]
    .rolling(window=4, min_periods=2)
    .var()
    .reset_index(level=[0,1,2], drop=True)
)
final_rca_frame["dominance_instability_flag"] = (
    final_rca_frame["dominance_score_var_4"] > 0.25
)


# 8) RCA Mode

conditions = [
    # 1) No RCA: UOM change or duplicate SKU–plant
    (
        final_rca_frame["uom_change_flag"].fillna(False).astype(bool) |
        final_rca_frame["duplicate_sku_plant_flag"].fillna(False).astype(bool)
    ),
    # 2) Entry / Exit RCA
    (
        final_rca_frame["sku_presence_class"]
        .isin(["ENTERED", "OUT"])
        .fillna(False)
    ),
    # 3) Limited RCA: insufficient history
    (
        ~final_rca_frame["has_sufficient_6periods"]
    ),
    # 4) Full temporal RCA: stable SKU
    (
        final_rca_frame["sku_presence_class"]
        .eq("STABLE")
        .fillna(False)
    )
]

choices = [
    "NO_RCA",
    "ENTRY_EXIT_RCA",
    "LIMITED_RCA",
    "FULL_TEMPORAL_RCA"
]

final_rca_frame["rca_mode"] = np.select(
    conditions,
    choices,
    default="LIMITED_RCA"
)

### Noise vs Signal Determination 

In [138]:
# Sorting

final_rca_frame = final_rca_frame.sort_values(
    ["material", "plant","date", "snapshot_date"]
)

#  1) Business materiality (value-based)   
# How big the change is in value terms, relative to prior fip.
final_rca_frame["relative_fip_impact"] = np.where(
    final_rca_frame["total_cost_prev"] > 0,
    final_rca_frame["total_fip_change"].abs() /
    final_rca_frame["total_cost_prev"],
    np.nan
)

#### Persistance

In [139]:
# Temporal persistence (directional consistency)

final_rca_frame["quantity_change_sign"] = np.sign(
    final_rca_frame["delta_quantity"]
) 

def persistence_score(series):
    score = []
    current = 0
    prev = 0
    for v in series:
        if v == 0 or pd.isna(v):
            current = 0
        elif v == prev:
            current += 1
        else:
            current = 1
        score.append(current)
        prev = v
    return score

final_rca_frame["quantity_persistence_score"] = (
    final_rca_frame
    .groupby(["material", "plant","date"])["quantity_change_sign"]
    .transform(persistence_score)
)


In [140]:
# Outlier Identification

# PARAMETERS

ROLLING_WINDOW = 12
MIN_PERIODS = 6

EXTREME_PCT_CHANGE = 0.5      # 50%
LOW_LEVEL_FLOOR = 0.05        # collapse threshold (5%)
HIGH_LEVEL_MULT = 10         # spike threshold (10x)

ROBUST_Z_THRESHOLD = 3
MIN_ZSCORE_POINTS = 3
PCTL_FALLBACK = 0.95

# 1) Rolling median of quantity (baseline scale)

final_rca_frame["quantity_rolling_median_12w"] = (
    final_rca_frame
        .groupby(["material", "plant","date"])["quantity_curr"]
        .transform(
            lambda x: x.shift(1)
                      .rolling(ROLLING_WINDOW, min_periods=MIN_PERIODS)
                      .median()
        )
)

# 2) Structural extreme change (scale-based)

final_rca_frame["is_structural_extreme_qty"] = (
    (final_rca_frame["delta_quantity_pct"].abs() >= EXTREME_PCT_CHANGE) &
    (
        (final_rca_frame["quantity_curr"] <=
         LOW_LEVEL_FLOOR * final_rca_frame["quantity_rolling_median_12w"]) |
        (final_rca_frame["quantity_curr"] >=
         HIGH_LEVEL_MULT * final_rca_frame["quantity_rolling_median_12w"])
    )
)



# 3) CLEAN delta series (absolute, exclude structural extremes)

final_rca_frame["clean_delta_quantity"] = final_rca_frame["delta_quantity"]

final_rca_frame.loc[
    final_rca_frame["is_structural_extreme_qty"],
    "clean_delta_quantity"
] = np.nan



# 4) Robust rolling MAD of absolute delta quantity

final_rca_frame["delta_qty_rolling_mad_12w"] = (
    final_rca_frame
        .groupby(["material", "plant","date"])["clean_delta_quantity"]
        .transform(
            lambda x: x.shift(1)
                      .rolling(ROLLING_WINDOW, min_periods=MIN_PERIODS)
                      .apply(
                          lambda s: np.nanmedian(
                              np.abs(s - np.nanmedian(s))
                          ),
                          raw=True
                      )
        )
)

# 5) Robust MAD-based z-score (absolute delta)

def robust_zscore(series, min_points=MIN_ZSCORE_POINTS):
    valid = series.dropna()

    if len(valid) < min_points:
        return pd.Series(np.nan, index=series.index)

    median = np.nanmedian(valid)
    mad = np.nanmedian(np.abs(valid - median))

    if mad == 0 or np.isnan(mad):
        return pd.Series(np.nan, index=series.index)

    return (series - median) / (1.4826 * mad)


final_rca_frame["quantity_z_scr"] = (
    final_rca_frame
        .groupby(["material", "plant","date"])["clean_delta_quantity"]
        .transform(lambda x: robust_zscore(x.shift(1)))
)



# 6) Z-score availability (CORRECT, LOCAL gating)

final_rca_frame["qty_z_hist_count"] = (
    final_rca_frame
        .groupby(["material", "plant","date"])["clean_delta_quantity"]
        .transform(
            lambda x: x.shift(1)
                      .rolling(ROLLING_WINDOW, min_periods=1)
                      .count()
        )
)

final_rca_frame["has_quantity_zscore"] = (
    (final_rca_frame["qty_z_hist_count"] >= MIN_ZSCORE_POINTS) &
    (final_rca_frame["delta_qty_rolling_mad_12w"] > 0)
)



# 7) Statistical outlier (MAD-based)

final_rca_frame["is_statistical_outlier_qty"] = (
    final_rca_frame["has_quantity_zscore"] &
    (final_rca_frame["quantity_z_scr"].abs() > ROBUST_Z_THRESHOLD)
)


# 8) Percentile fallback (ONLY when z-score unavailable)
final_rca_frame["qty_abs_pct_threshold"] = (
    final_rca_frame
        .groupby(["material", "plant","date"])["clean_delta_quantity"]
        .transform(
            lambda x: x.shift(1)
                      .abs()
                      .quantile(PCTL_FALLBACK)
        )
)

# Fallback scale must be valid (non-zero)
final_rca_frame["has_valid_pct_scale"] = (
    final_rca_frame["qty_abs_pct_threshold"] > 0
)

final_rca_frame["is_pct_outlier_qty"] = (
    final_rca_frame["clean_delta_quantity"].abs() >
    final_rca_frame["qty_abs_pct_threshold"]
)

# 9) FINAL quantity outlier flag (hierarchical & SAFE)

final_rca_frame["is_quantity_outlier"] = (
    final_rca_frame["is_structural_extreme_qty"] |
    np.where(
        final_rca_frame["has_quantity_zscore"],
        final_rca_frame["is_statistical_outlier_qty"],
        final_rca_frame["has_valid_pct_scale"] &
        final_rca_frame["is_pct_outlier_qty"]
    )
)


#### Change Point Detection

In [141]:
# Change Point Detection

# PARAMETERS

CP_LONG_WINDOW = 12
CP_LONG_MIN = 6
CP_SHORT_WINDOW = 6
CP_SHORT_MIN = 3

CP_TREND_STRENGTH_THRESHOLD = 2
CP_FALLBACK_STRENGTH_THRESHOLD = 5   # conservative
MIN_PERSISTENCE_CP = 2


# 1) Rolling average of ABSOLUTE delta quantity (long window)

final_rca_frame["rolling_avg_delta_qty_12w"] = (
    final_rca_frame
        .groupby(["material", "plant","date"])["delta_quantity"]
        .transform(
            lambda x: x.shift(1)
                      .rolling(CP_LONG_WINDOW, min_periods=CP_LONG_MIN)
                      .mean()
        )
)


# 2) rolling MAD of ABSOLUTE delta quantity

final_rca_frame["delta_qty_rolling_mad_12w"] = (
    final_rca_frame
        .groupby(["material", "plant","date"])["delta_quantity"]
        .transform(
            lambda x: x.shift(1)
                      .rolling(CP_LONG_WINDOW, min_periods=CP_LONG_MIN)
                      .apply(
                          lambda s: np.nanmedian(
                              np.abs(s - np.nanmedian(s))
                          ),
                          raw=True
                      )
        )
)

# 3) Primary trend strength (MAD-based)

final_rca_frame["trend_strength"] = (
    final_rca_frame["rolling_avg_delta_qty_12w"].abs() /
    (1.4826 * final_rca_frame["delta_qty_rolling_mad_12w"])
)

# Invalidate degenerate cases
final_rca_frame.loc[
    (final_rca_frame["delta_qty_rolling_mad_12w"] <= 0) |
    (final_rca_frame["delta_qty_rolling_mad_12w"].isna()),
    "trend_strength"
] = np.nan

final_rca_frame["has_valid_cp_strength"] = (
    final_rca_frame["trend_strength"].notna()
)


# 4) GENERIC fallback scale (never collapses)

final_rca_frame["delta_qty_rolling_median_abs_12w"] = (
    final_rca_frame
        .groupby(["material", "plant","date"])["delta_quantity"]
        .transform(
            lambda x: x.shift(1)
                      .rolling(CP_LONG_WINDOW, min_periods=CP_LONG_MIN)
                      .apply(lambda s: np.nanmedian(np.abs(s)), raw=True)
        )
)

final_rca_frame["fallback_change_strength"] = (
    final_rca_frame["rolling_avg_delta_qty_12w"].abs() /
    final_rca_frame["delta_qty_rolling_median_abs_12w"]
)

# Invalidate fallback when scale unusable
final_rca_frame.loc[
    (final_rca_frame["delta_qty_rolling_median_abs_12w"] <= 0) |
    (final_rca_frame["delta_qty_rolling_median_abs_12w"].isna()),
    "fallback_change_strength"
] = np.nan

final_rca_frame["use_fallback_cp"] = (
    final_rca_frame["trend_strength"].isna() &
    final_rca_frame["fallback_change_strength"].notna()
)

# 5) Short-window trend confirmation (ABSOLUTE delta)
final_rca_frame["rolling_avg_delta_qty_6w"] = (
    final_rca_frame
        .groupby(["material", "plant","date"])["delta_quantity"]
        .transform(
            lambda x: x.shift(1)
                      .rolling(CP_SHORT_WINDOW, min_periods=CP_SHORT_MIN)
                      .mean()
        )
)

final_rca_frame["trend_confirmed"] = (
    np.sign(final_rca_frame["rolling_avg_delta_qty_12w"]) ==
    np.sign(final_rca_frame["rolling_avg_delta_qty_6w"])
)


# 6) FINAL generic change-point detection
final_rca_frame["change_point_detected"] = (
    (final_rca_frame["quantity_persistence_score"] >= MIN_PERSISTENCE_CP) &
    final_rca_frame["trend_confirmed"] &
    (
        # Primary MAD-based path
        (
            final_rca_frame["has_valid_cp_strength"] &
            (final_rca_frame["trend_strength"] > CP_TREND_STRENGTH_THRESHOLD)
        )
        |
        # Fallback absolute-scale path
        (
            final_rca_frame["use_fallback_cp"] &
            (final_rca_frame["fallback_change_strength"] > CP_FALLBACK_STRENGTH_THRESHOLD)
        )
    )
)



#### Regime Stability

In [142]:
# Regime stability - validate the change point with stability

REGIME_STABILITY_WINDOW = 3        # how many snapshots must settle
REGIME_STABILITY_TOLERANCE = 0.2   # ±20% band around new level

# 1) Reference "new level" after change

final_rca_frame["post_cp_level"] = (
    final_rca_frame
        .groupby(["material", "plant","date"])["quantity_curr"]
        .shift(1)
)


# 2) Within-band check
final_rca_frame["within_new_regime_band"] = (
    final_rca_frame["quantity_curr"].between(
        final_rca_frame["post_cp_level"] * (1 - REGIME_STABILITY_TOLERANCE),
        final_rca_frame["post_cp_level"] * (1 + REGIME_STABILITY_TOLERANCE)
    )
)


# 3) Rolling stability confirmation
final_rca_frame["regime_stability_score"] = (
    final_rca_frame
        .groupby(["material", "plant","date"])["within_new_regime_band"]
        .transform(
            lambda x: x.shift(-1)   # look forward (post-change validation)
                      .rolling(REGIME_STABILITY_WINDOW, min_periods=REGIME_STABILITY_WINDOW)
                      .sum()
        )
)

final_rca_frame["is_regime_stable"] = (
    final_rca_frame["regime_stability_score"] >= REGIME_STABILITY_WINDOW
)


final_rca_frame["effective_change_point"] = (
    final_rca_frame["change_point_detected"] &
    final_rca_frame["is_regime_stable"]
)


#### Noise & Signal Flagging

In [143]:
# BASE METHOD 

final_rca_frame["is_noise_base"] = (
    final_rca_frame["is_quantity_outlier"] &
    (~final_rca_frame["effective_change_point"])
)

# insufficient data → cannot classify as noise yet
final_rca_frame.loc[
    ~final_rca_frame["has_sufficient_6periods"],
    "is_noise_base"
] = False

final_rca_frame["is_signal_base"] = ~final_rca_frame["is_noise_base"]


In [144]:
# 2nd Layer Flaging

final_rca_frame["is_explainable"] = (
    # data quality must be OK
    (~final_rca_frame["uom_change_flag"]) &
    (~final_rca_frame["duplicate_sku_plant_flag"]) 
    # &
    # # must not be lifecycle noise -- needs data backed thresholds to handle different edge cases
    # (final_rca_frame["sku_presence_class"] == "STABLE")
)

final_rca_frame["is_noise_governed"] = (
    final_rca_frame["is_noise_base"] |
    (~final_rca_frame["is_explainable"])
)

final_rca_frame["is_signal_governed"] = (
    ~final_rca_frame["is_noise_governed"]
)

final_rca_frame["noise_reason"] = None

final_rca_frame.loc[
    final_rca_frame["uom_change_flag"] |
    final_rca_frame["duplicate_sku_plant_flag"],
    "noise_reason"
] = "DATA_QUALITY"

final_rca_frame.loc[
    final_rca_frame["noise_reason"].isna() &
    final_rca_frame["is_noise_base"],
    "noise_reason"
] = "STATISTICAL_NOISE"


## Price Outliers Addition

In [145]:
PRICE_OUTLIER_PCT_THRESHOLD = 1   # 100% change

final_rca_frame["is_price_outlier"] = (
    final_rca_frame["delta_cost_per_unit_pct"].abs() >= PRICE_OUTLIER_PCT_THRESHOLD
)

final_rca_frame["price_event_type"] = "NO_PRICE_EVENT"

# Price-driven RCA case (quantity is NOT the driver)
final_rca_frame.loc[
    final_rca_frame["is_signal_governed"] &
    final_rca_frame["is_price_outlier"] &
    (~final_rca_frame["is_quantity_outlier"]),
    "price_event_type"
] = "PRICE_PRIMARY"

# Mixed RCA case (price amplifies a quantity-driven change)
final_rca_frame.loc[
    final_rca_frame["is_signal_governed"] &
    final_rca_frame["is_price_outlier"] &
    final_rca_frame["is_quantity_outlier"],
    "price_event_type"
] = "PRICE_SECONDARY"


#### Brand + Snapshot Date Level Top 10 SKUs

In [146]:
final_rca_frame["abs_fip_change"] = final_rca_frame["total_fip_change"].abs()

final_rca_frame["snapshot_signal_rank"] = (
    final_rca_frame
    .where(final_rca_frame["is_signal_governed"])
    .groupby(["snapshot_date", "corporate_brand","date"])["abs_fip_change"]
    .rank(method="first", ascending=False)
)
final_rca_frame["is_top10_contributor"] = (
    final_rca_frame["snapshot_signal_rank"] <= 10
)

final_rca_frame = final_rca_frame.drop(["snapshot_signal_rank","abs_fip_change"],axis=1)

In [147]:
final_rca_frame.head()

,corporate_brand,material_type,development_lifecycle_status,enterprise_category,enterprise_sub_category,dosage_form_parent,corp_brand_id,network_or_business_unit,profit_center,brand_name,...,effective_change_point,is_noise_base,is_signal_base,is_explainable,is_noise_governed,is_signal_governed,noise_reason,is_price_outlier,price_event_type,is_top10_contributor
0,None,None,None,None,None,None,None,BIOLOGICS,<NA>,<NA>,...,False,False,True,True,False,True,None,False,NO_PRICE_EVENT,False
1,None,None,None,None,None,None,None,BIOLOGICS,<NA>,<NA>,...,False,False,True,True,False,True,None,False,NO_PRICE_EVENT,False
2,None,None,None,None,None,None,None,BIOLOGICS,<NA>,<NA>,...,False,False,True,True,False,True,None,False,NO_PRICE_EVENT,False
3,None,None,None,None,None,None,None,BIOLOGICS,<NA>,<NA>,...,False,False,True,True,False,True,None,False,NO_PRICE_EVENT,False
4,None,None,None,None,None,None,None,BIOLOGICS,<NA>,<NA>,...,False,False,True,True,False,True,None,False,NO_PRICE_EVENT,False


In [148]:
# brands_to_keep = [
#     "All Other Pharmaceut"
#     # ,
#     #"ABRAXANE"
#     #,
#     #"REVLIMID"
# ]
# dates_to_keep = ["202612"] #, "202712", "202812"]

# filtered_df = final_rca_frame[
#     final_rca_frame["corporate_brand"].isin(brands_to_keep) &
#     final_rca_frame["date"].isin(dates_to_keep)
# ]

In [149]:
# final_rca_frame.loc[
#     final_rca_frame["corporate_brand"].str.contains(
#         "All Other Pharmaceut", case=False, na=False
#     ),
#     "corporate_brand"
# ].unique()


In [150]:
final_rca_frame.columns

Index(['corporate_brand', 'material_type', 'development_lifecycle_status',
       'enterprise_category', 'enterprise_sub_category', 'dosage_form_parent',
       'corp_brand_id', 'network_or_business_unit', 'profit_center',
       'brand_name',
       ...
       'effective_change_point', 'is_noise_base', 'is_signal_base',
       'is_explainable', 'is_noise_governed', 'is_signal_governed',
       'noise_reason', 'is_price_outlier', 'price_event_type',
       'is_top10_contributor'],
      dtype='object', length=114)

# Brand Aggregations

### Data Prep

In [176]:
sku_df = final_rca_frame.copy()

# Define SKU correctly: material–plant
sku_df["sku_id"] = (
    sku_df["material"].astype(str) + "||" +
    sku_df["plant"].astype(str)
)

# Normalize boolean (critical for stability)
sku_df["is_signal_governed"] = (
    sku_df["is_signal_governed"]
        .fillna(False)
        .astype(bool)
)

# Helper columns
sku_df["abs_sku_impact"] = sku_df["total_fip_change"].abs()

sku_df["signal_impact"] = np.where(
    sku_df["is_signal_governed"],
    sku_df["abs_sku_impact"],
    0.0
)

sku_df["conflict_evaluable"] = (
    sku_df["is_signal_governed"] &
    sku_df["quantity_impact"].notna() &
    sku_df["cost_impact"].notna()
)

sku_df["driver_conflict"] = np.where(
    sku_df["conflict_evaluable"],
    np.sign(sku_df["quantity_impact"]) != np.sign(sku_df["cost_impact"]),
    np.nan
)

sku_df["weighted_dom_component"] = (
    sku_df["dominance_score"] * sku_df["abs_sku_impact"]
)

sku_df["signal_structural_cp"] = (
    sku_df["is_signal_governed"] &
    sku_df["effective_change_point"]
)


In [177]:
sku_df.shape

(703000, 121)

# Brand Level Calculations

In [178]:
# 1. BRAND-LEVEL BASE AGG

brand_df = (
    sku_df
    .groupby(
        ["corp_brand_id", "corporate_brand", "date", "snapshot_date"],
        as_index=False
    )
    .agg(
        ΔFIP=("total_fip_change", "sum"),
        Prior_FIP=("total_cost_prev", "sum"),
        Abs_Impact=("abs_sku_impact", "sum"),

        Total_SKUs=("sku_id", "nunique"),
        Signal_SKUs=("is_signal_governed", "sum"),
        Signal_Impact=("signal_impact", "sum"),

        Qty_Impact=("quantity_impact", "sum"),
        Cost_Impact=("cost_impact", "sum"),

        Weighted_Dom_Num=("weighted_dom_component", "sum"),
        Weighted_Dom_Den=("abs_sku_impact", "sum"),

        Driver_Conflict_Count=("driver_conflict", "sum"),
        Avg_Persistence=("quantity_persistence_score", "mean"),
        Structural_CP_Count=("signal_structural_cp", "sum"),
    )
)


# 2. SAFE DIVISION HELPERS


def safe_div(n, d):
    return np.divide(
        n,
        d,
        out=np.full_like(n, np.nan, dtype=float),
        where=d != 0
    )



# 3. DERIVED BRAND METRICS


brand_df["FIP_pct_change"] = safe_div(
    brand_df["ΔFIP"],
    brand_df["Prior_FIP"]
)

brand_df["Net_vs_Abs_Ratio"] = safe_div(
    brand_df["ΔFIP"].abs(),
    brand_df["Abs_Impact"]
)

brand_df["Signal_SKU_%"] = safe_div(
    brand_df["Signal_SKUs"],
    brand_df["Total_SKUs"]
)

brand_df["Impact_from_Signal_%"] = safe_div(
    brand_df["Signal_Impact"],
    brand_df["Abs_Impact"]
)

brand_df["Qty_Impact_%"] = safe_div(
    brand_df["Qty_Impact"],
    brand_df["ΔFIP"]
)

brand_df["Cost_Impact_%"] = safe_div(
    brand_df["Cost_Impact"],
    brand_df["ΔFIP"]
)

brand_df["Weighted_Dominance"] = safe_div(
    brand_df["Weighted_Dom_Num"],
    brand_df["Weighted_Dom_Den"]
)

brand_df["Driver_Conflict_%"] = safe_div(
    brand_df["Driver_Conflict_Count"],
    brand_df["Signal_SKUs"]
)

brand_df["Structural_Shift_Index"] = safe_div(
    brand_df["Structural_CP_Count"],
    brand_df["Signal_SKUs"]
) * 100

brand_df["Ownership_Clarity_Index"] = (
    brand_df["Qty_Impact_%"] - brand_df["Cost_Impact_%"]
).abs()

brand_df["level"] = "BRAND"


# 4. SKU IMPACT PRE-AGG (ONCE)


sku_impact = (
    sku_df
    .groupby(
        ["corp_brand_id", "corporate_brand", "date", "snapshot_date", "sku_id"],
        as_index=False
    )["abs_sku_impact"]
    .sum()
)

sku_impact = sku_impact.sort_values(
    ["corp_brand_id", "corporate_brand", "date", "snapshot_date", "abs_sku_impact"],
    ascending=[True, True, True, True, False]
)


# 5. TOP-K SKU IMPACTS (ONE PASS)

topk = (
    sku_impact
    .groupby(
        ["corp_brand_id", "corporate_brand", "date", "snapshot_date"]
    )
    .apply(
        lambda x: pd.Series({
            "Top1_SKU_Impact": x["abs_sku_impact"].head(1).sum(),
            "Top5_SKU_Impact": x["abs_sku_impact"].head(5).sum(),
            "Top10_SKU_Impact": x["abs_sku_impact"].head(10).sum(),
        })
    )
    .reset_index()
)

brand_df = brand_df.merge(
    topk,
    on=["corp_brand_id", "corporate_brand", "date", "snapshot_date"],
    how="left"
)

brand_df["Top_1_SKU_%"] = safe_div(
    brand_df["Top1_SKU_Impact"],
    brand_df["Abs_Impact"]
)

brand_df["Top_5_SKU_%"] = safe_div(
    brand_df["Top5_SKU_Impact"],
    brand_df["Abs_Impact"]
)

brand_df["Top_10_SKU_%"] = safe_div(
    brand_df["Top10_SKU_Impact"],
    brand_df["Abs_Impact"]
)


# 6. TOP PLANT CONCENTRATION

plant_impact = (
    sku_df
    .groupby(
        ["corp_brand_id", "corporate_brand", "date", "snapshot_date", "plant"],
        as_index=False
    )["abs_sku_impact"]
    .sum()
)

top_plant = (
    plant_impact
    .sort_values(
        ["corp_brand_id", "corporate_brand", "date", "snapshot_date", "abs_sku_impact"],
        ascending=[True, True, True, True, False]
    )
    .groupby(
        ["corp_brand_id", "corporate_brand", "date", "snapshot_date"]
    )
    .head(1)
    .rename(columns={"abs_sku_impact": "Top_Plant_Impact"})
)

brand_df = brand_df.merge(
    top_plant[
        ["corp_brand_id", "corporate_brand", "date", "snapshot_date", "Top_Plant_Impact"]
    ],
    on=["corp_brand_id", "corporate_brand", "date", "snapshot_date"],
    how="left"
)

brand_df["Plant_Concentration_%"] = safe_div(
    brand_df["Top_Plant_Impact"],
    brand_df["Abs_Impact"]
)

# 7. COMPOSITE SCORES

brand_df["Actionability_Index"] = (
    brand_df["Impact_from_Signal_%"] *
    brand_df["Top_5_SKU_%"]
)

brand_df["Explainability_Score"] = (
    0.4 * brand_df["Top_5_SKU_%"] +
    0.4 * brand_df["Signal_SKU_%"] +
    0.2 * (brand_df["Avg_Persistence"] / 6)
)


/tmp/ipykernel_422638/160256252.py:124: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


In [179]:
brand_df.columns

Index(['corp_brand_id', 'corporate_brand', 'date', 'snapshot_date', 'ΔFIP',
       'Prior_FIP', 'Abs_Impact', 'Total_SKUs', 'Signal_SKUs', 'Signal_Impact',
       'Qty_Impact', 'Cost_Impact', 'Weighted_Dom_Num', 'Weighted_Dom_Den',
       'Driver_Conflict_Count', 'Avg_Persistence', 'Structural_CP_Count',
       'FIP_pct_change', 'Net_vs_Abs_Ratio', 'Signal_SKU_%',
       'Impact_from_Signal_%', 'Qty_Impact_%', 'Cost_Impact_%',
       'Weighted_Dominance', 'Driver_Conflict_%', 'Structural_Shift_Index',
       'Ownership_Clarity_Index', 'level', 'Top1_SKU_Impact',
       'Top5_SKU_Impact', 'Top10_SKU_Impact', 'Top_1_SKU_%', 'Top_5_SKU_%',
       'Top_10_SKU_%', 'Top_Plant_Impact', 'Plant_Concentration_%',
       'Actionability_Index', 'Explainability_Score'],
      dtype='object')

### Brand Level Calculations along with Dominant Level

In [180]:
LEVEL_MAP = {
    "material_type": ["material_type"],
    "dosage_form_parent": ["dosage_form_parent"],
    "enterprise_category": ["enterprise_category"],
    #"development_lifecycle_status": ["development_lifecycle_status"],
    #"material_group": ["material_group"],
    "material": ["material"],
    "nodetype": ["nodetype"],
    "Plant Type": ["Plant Type"],
    "plant": ["plant"],
    "material × plant": ["material", "plant"]  # SKU level
}

LEVEL_ORDER = list(LEVEL_MAP.keys())


In [181]:
# Concentration + HHI Calculation

def compute_level_concentration(sku_brand_df, group_cols, brand_delta_fip):

    if sku_brand_df.empty or brand_delta_fip == 0:
        return None

    grp = (
        sku_brand_df
        .groupby(group_cols, as_index=False)
        .agg(Group_ΔFIP=("total_fip_change", "sum"))
    )

    grp["share"] = grp["Group_ΔFIP"].abs() / abs(brand_delta_fip)
    grp = grp.sort_values("share", ascending=False)

    hhi = (grp["share"] ** 2).sum()

    return {
        "max_share": grp["share"].iloc[0],
        "hhi": hhi,
        "top_entities": grp[group_cols].head(2).to_dict("records"),
        "shares": grp["share"].head(2).values
    }


In [182]:
# level selection

def select_winning_level(sku_brand_df, brand_delta_fip):

    best = None

    for level in LEVEL_ORDER:

        res = compute_level_concentration(
            sku_brand_df,
            LEVEL_MAP[level],
            brand_delta_fip
        )

        if res is None:
            continue

        if best is None:
            best = {"level": level, **res}
            continue

        # choose level with higher concentration (HHI priority)
        if res["hhi"] > best["hhi"]:
            best = {"level": level, **res}

    return best


In [183]:
# Compute Entity Metrics

def compute_entity_metrics(df):

    out = {}

    out["ΔFIP"] = df["total_fip_change"].sum()
    out["Prior_FIP"] = df["total_cost_prev"].sum()
    out["Abs_Impact"] = df["abs_sku_impact"].sum()

    out["Total_SKUs"] = df["sku_id"].nunique()
    out["Signal_SKUs"] = df["is_signal_governed"].sum()
    out["Signal_Impact"] = df["signal_impact"].sum()

    out["Qty_Impact"] = df["quantity_impact"].sum()
    out["Cost_Impact"] = df["cost_impact"].sum()

    out["Signal_SKU_%"] = safe_div(out["Signal_SKUs"], out["Total_SKUs"])
    out["Impact_from_Signal_%"] = safe_div(out["Signal_Impact"], out["Abs_Impact"])
    out["Qty_Impact_%"] = safe_div(out["Qty_Impact"], out["ΔFIP"])
    out["Cost_Impact_%"] = safe_div(out["Cost_Impact"], out["ΔFIP"])

    out["Weighted_Dominance"] = safe_div(
        df["weighted_dom_component"].sum(),
        df["abs_sku_impact"].sum()
    )

    out["Avg_Persistence"] = df["quantity_persistence_score"].mean()

    out["Structural_Shift_Index"] = safe_div(
        df["signal_structural_cp"].sum(),
        out["Signal_SKUs"]
    ) * 100

    out["Ownership_Clarity_Index"] = abs(
        out["Qty_Impact_%"] - out["Cost_Impact_%"]
    )

    return out


In [184]:
dominant_rows = []

for _, brand_row in brand_df.iterrows():

    brand_delta_fip = brand_row["ΔFIP"]

    sku_brand_df = sku_df.loc[
        (sku_df["corp_brand_id"] == brand_row["corp_brand_id"]) &
        (sku_df["date"] == brand_row["date"]) &
        (sku_df["snapshot_date"] == brand_row["snapshot_date"])
    ]

    if sku_brand_df.empty:
        continue

    selection = select_winning_level(
        sku_brand_df,
        brand_delta_fip
    )

    if selection is None:
        continue

    winning_level = selection["level"]
    winning_cols = LEVEL_MAP[winning_level]
    hhi = selection["hhi"]

    # Decide how many groups to surface
    if hhi > 0.7:
        groups_to_surface = selection["top_entities"][:1]
        regime = "Single"
    elif 0.3 <= hhi <= 0.7:
        groups_to_surface = selection["top_entities"][:2]
        regime = "Dual"
    else:
        # Diffuse story → skip entity rows
        continue

    for entity_dict in groups_to_surface:

        entity_mask = (
            (sku_df["corp_brand_id"] == brand_row["corp_brand_id"]) &
            (sku_df["date"] == brand_row["date"]) &
            (sku_df["snapshot_date"] == brand_row["snapshot_date"])
        )

        for col, val in entity_dict.items():
            entity_mask &= (sku_df[col] == val)

        sub_df = sku_df.loc[entity_mask]

        if sub_df.empty:
            continue

        metrics = compute_entity_metrics(sub_df)

        metrics.update({
            "corp_brand_id": brand_row["corp_brand_id"],
            "corporate_brand": (
                f"{brand_row['corporate_brand']}_"
                + "_".join(str(v) for v in entity_dict.values())
            ),
            "date": brand_row["date"],
            "snapshot_date": brand_row["snapshot_date"],
            "level": f"BRAND_{winning_level.upper()}",
            "Entity_Impact_onBrandlevel_%": (
                abs(metrics["ΔFIP"]) / abs(brand_delta_fip)
                if brand_delta_fip != 0 else np.nan
            ),
            "Concentration_Regime": regime,
            "Winning_Level_HHI": hhi
        })

        dominant_rows.append(metrics)


In [185]:
dominant_entity_df = pd.DataFrame(dominant_rows)

final_agg_df = pd.concat(
    [brand_df, dominant_entity_df],
    ignore_index=True
)

In [186]:
final_agg_df.head()

,corp_brand_id,corporate_brand,date,snapshot_date,ΔFIP,Prior_FIP,Abs_Impact,Total_SKUs,Signal_SKUs,Signal_Impact,...,Top_1_SKU_%,Top_5_SKU_%,Top_10_SKU_%,Top_Plant_Impact,Plant_Concentration_%,Actionability_Index,Explainability_Score,Entity_Impact_onBrandlevel_%,Concentration_Regime,Winning_Level_HHI
0,00000nan,nan,202612,2025-07-18,0.000000,0.000000,0.000000,1088,1088,0.000000,...,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
1,00000nan,nan,202612,2025-07-25,0.000000,30261.127717,0.000000,1089,16,0.000000,...,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
2,00000nan,nan,202612,2025-08-01,0.926509,30261.127717,0.926509,1089,15,0.926509,...,1.000000,1.0,1.0,0.926509,1.000000,1.0,0.406061,NaN,NaN,NaN
3,00000nan,nan,202612,2025-08-08,1747.595640,20998.820043,1958.441240,1078,19,1958.441240,...,0.946170,1.0,1.0,1958.440240,0.999999,1.0,0.409617,NaN,NaN,NaN
4,00000nan,nan,202612,2025-08-15,-1927.052530,22746.415683,1927.052530,1093,15,1927.052530,...,0.961581,1.0,1.0,1927.051530,0.999999,1.0,0.408265,NaN,NaN,NaN


In [187]:
final_agg_df['level'].value_counts()

level
BRAND                        6133
BRAND_MATERIAL_TYPE           400
BRAND_DOSAGE_FORM_PARENT      224
BRAND_MATERIAL × PLANT        223
BRAND_NODETYPE                170
BRAND_PLANT TYPE              159
BRAND_PLANT                   150
BRAND_MATERIAL                118
BRAND_ENTERPRISE_CATEGORY     102
Name: count, dtype: int64

In [188]:
final_agg_df[(final_agg_df['snapshot_date'] == '2026-01-09') & (final_agg_df['corp_brand_id'] == '03302101')]

,corp_brand_id,corporate_brand,date,snapshot_date,ΔFIP,Prior_FIP,Abs_Impact,Total_SKUs,Signal_SKUs,Signal_Impact,...,Top_1_SKU_%,Top_5_SKU_%,Top_10_SKU_%,Top_Plant_Impact,Plant_Concentration_%,Actionability_Index,Explainability_Score,Entity_Impact_onBrandlevel_%,Concentration_Regime,Winning_Level_HHI
4626,03302101,REVLIMID,202612,2026-01-09,-1.491973e+09,1.523524e+09,1.493309e+09,842,731,3.269411e+06,...,0.992682,0.995658,0.99683,1.485222e+09,0.994584,0.00218,0.755152,NaN,NaN,NaN
7132,03302101,REVLIMID_HALB,202612,2026-01-09,-1.485717e+09,1.493397e+09,1.486367e+09,112,92,1.168630e+06,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.995807,Single,0.991649


# Validation

In [189]:
def debug_one_snapshot(brand_id, date, snapshot_date):

    brand_row = brand_df.loc[
        (brand_df["corp_brand_id"] == brand_id) &
        (brand_df["date"] == date) &
        (brand_df["snapshot_date"] == snapshot_date)
    ]

    if brand_row.empty:
        print("Brand snapshot not found.")
        return

    brand_delta_fip = brand_row["ΔFIP"].iloc[0]

    print("="*70)
    print("Brand ΔFIP:", brand_delta_fip)

    sku_brand_df = sku_df.loc[
        (sku_df["corp_brand_id"] == brand_id) &
        (sku_df["date"] == date) &
        (sku_df["snapshot_date"] == snapshot_date)
    ]

    print("Total SKUs:", sku_brand_df["sku_id"].nunique())
    print("="*70)

    results = []

    for level in LEVEL_ORDER:

        res = compute_level_concentration(
            sku_brand_df,
            LEVEL_MAP[level],
            brand_delta_fip
        )

        if res is None:
            continue

        print("\nLEVEL:", level)
        print("Max Share:", round(res["max_share"], 4))
        print("HHI:", round(res["hhi"], 4))
        print("Top Entity:", res["top_entities"][0])

        results.append({
            "level": level,
            "max_share": res["max_share"],
            "hhi": res["hhi"]
        })

    print("\n" + "="*70)

    results_df = pd.DataFrame(results).sort_values("hhi", ascending=False)

    print("\nLEVELS SORTED BY HHI (highest concentration first):")
    print(results_df)

    print("="*70)

    return results_df


In [190]:
check = (
    sku_df
    .groupby("material_group")["material_type"]
    .nunique()
)

print(check.value_counts())

material_type
1    52
2    30
3     9
4     7
5     1
Name: count, dtype: int64


In [191]:
debug_one_snapshot(
    brand_id="03302101",
    date="202612",
    snapshot_date="2026-01-09"
)

Brand ΔFIP: -1491973166.7871447
Total SKUs: 842

LEVEL: material_type
Max Share: 0.9958
HHI: 0.9916
Top Entity: {'material_type': 'HALB'}

LEVEL: dosage_form_parent
Max Share: 0.9956
HHI: 0.9912
Top Entity: {'dosage_form_parent': 'nan'}

LEVEL: enterprise_category
Max Share: 0.9956
HHI: 0.9912
Top Entity: {'enterprise_category': 'API'}

LEVEL: material
Max Share: 0.9936
HHI: 0.9872
Top Entity: {'material': '1456877'}

LEVEL: nodetype
Max Share: 0.9957
HHI: 0.9914
Top Entity: {'nodetype': 'T'}

LEVEL: Plant Type
Max Share: 0.9953
HHI: 0.9906
Top Entity: {'Plant Type': 'External'}

LEVEL: plant
Max Share: 0.9953
HHI: 0.9907
Top Entity: {'plant': '2091'}

LEVEL: material × plant
Max Share: 0.9936
HHI: 0.9872
Top Entity: {'material': '1456877', 'plant': '2091'}


LEVELS SORTED BY HHI (highest concentration first):
                 level  max_share       hhi
0        material_type   0.995807  0.991649
4             nodetype   0.995696  0.991421
1   dosage_form_parent   0.995589  0.991217
2 

,level,max_share,hhi
0,material_type,0.995807,0.991649
4,nodetype,0.995696,0.991421
1,dosage_form_parent,0.995589,0.991217
2,enterprise_category,0.995582,0.991204
6,plant,0.995329,0.990683
5,Plant Type,0.995265,0.990574
3,material,0.993571,0.987190
7,material × plant,0.993571,0.987188


# Final Dataframe cleaning

In [ ]:
final_agg_df["level"] = (
    final_agg_df["level"]
        .str.replace(" × ", "_", regex=False)
        .str.replace(" ", "_", regex=False)
)

In [201]:
# Select & rename final columns

final_df = (
    final_agg_df
    .rename(columns={
        "corporate_brand": "entity",
        "FIP_pct_change": "FIP_change_pct",
        "Entity_Impact_onBrandlevel_%": "Entity_Abs_Impact_Share_pct",
        "Net_vs_Abs_Ratio": "Net_vs_Abs_pct",
        "Signal_SKU_%": "Signal_SKU_pct",
        "Impact_from_Signal_%": "Impact_from_Signal_pct",
        "Top_1_SKU_%": "Top_1_SKU_pct",
        "Top_5_SKU_%": "Top_5_SKU_pct",
        "Top_10_SKU_%": "Top_10_SKU_pct",
        "Plant_Concentration_%": "Plant_Concentration_pct",
        "Qty_Impact_%": "Qty_Impact_pct",
        "Cost_Impact_%": "Cost_Impact_pct",
        "Driver_Conflict_%": "Driver_Conflict_pct",
        "Actionability_Index": "Actionability_Index_pct",
        "Ownership_Clarity_Index": "Ownership_Clarity_Index_pct",
        "Explainability_Score": "Explainability_Score_pct",
        "Weighted_Dominance": "Driver_Weighted_Dominance",
        "Structural_Shift_Index": "Structural_Shift_Index_pct",
        "ΔFIP": "net_fip_change"
    })
    [
        [
            "corp_brand_id",
            "entity",
            "level",
            "date",
            "snapshot_date",
            "FIP_change_pct",
            "Abs_Impact",
            "net_fip_change",
            "Entity_Abs_Impact_Share_pct",
            "Net_vs_Abs_pct",
            "Signal_SKU_pct",
            "Impact_from_Signal_pct",
            "Top_1_SKU_pct",
            "Top_5_SKU_pct",
            "Top_10_SKU_pct",
            "Plant_Concentration_pct",
            "Qty_Impact_pct",
            "Cost_Impact_pct",
            "Driver_Weighted_Dominance",
            "Driver_Conflict_pct",
            "Avg_Persistence",
            "Structural_Shift_Index_pct",
            "Actionability_Index_pct",
            "Ownership_Clarity_Index_pct",
            "Explainability_Score_pct",
            "Concentration_Regime"
            
        ]
    ]
)


# Percentage columns (×100)
percent_cols = [
    "FIP_change_pct",
    "Entity_Abs_Impact_Share_pct",
    "Net_vs_Abs_pct",
    "Signal_SKU_pct",
    "Impact_from_Signal_pct",
    "Top_1_SKU_pct",
    "Top_5_SKU_pct",
    "Top_10_SKU_pct",
    "Plant_Concentration_pct",
    "Qty_Impact_pct",
    "Cost_Impact_pct",
    "Driver_Conflict_pct",
    "Actionability_Index_pct",
    "Ownership_Clarity_Index_pct",
    "Explainability_Score_pct",
]

final_df[percent_cols] = (
    final_df[percent_cols]
    .astype(float)
    .mul(100)
    .round(2)
)


# Numeric rounding

# Round most numeric columns to 2 decimals
numeric_round_2_cols = [
    "net_fip_change",
    "Abs_Impact",
    "Avg_Persistence",
    "Structural_Shift_Index_pct",
]

final_df[numeric_round_2_cols] = (
    final_df[numeric_round_2_cols]
    .astype(float)
    .round(2)
)

# Round Driver_Weighted_Dominance to 0 decimals
final_df["Driver_Weighted_Dominance"] = (
    final_df["Driver_Weighted_Dominance"]
    .astype(float)
    .round(0)
    .astype("Int64")
)


def remove_negative_zero(x):
    if isinstance(x, (int, float, np.floating)) and np.isclose(x, 0):
        return 0.0
    return x

cols_to_clean = [
    "Qty_Impact_pct",
    "Cost_Impact_pct",
    "Net_vs_Abs_pct",
    "Driver_Conflict_pct",
]

final_df[cols_to_clean] = final_df[cols_to_clean].applymap(remove_negative_zero)


/tmp/ipykernel_422638/594151990.py:124: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  final_df[cols_to_clean] = final_df[cols_to_clean].applymap(remove_negative_zero)


In [202]:
final_df.head()

,corp_brand_id,entity,level,date,snapshot_date,FIP_change_pct,Abs_Impact,net_fip_change,Entity_Abs_Impact_Share_pct,Net_vs_Abs_pct,...,Qty_Impact_pct,Cost_Impact_pct,Driver_Weighted_Dominance,Driver_Conflict_pct,Avg_Persistence,Structural_Shift_Index_pct,Actionability_Index_pct,Ownership_Clarity_Index_pct,Explainability_Score_pct,Concentration_Regime
0,00000nan,nan,BRAND,202612,2025-07-18,NaN,0.00,0.00,NaN,NaN,...,NaN,NaN,<NA>,0.00,0.00,0.0,NaN,NaN,NaN,NaN
1,00000nan,nan,BRAND,202612,2025-07-25,0.00,0.00,0.00,NaN,NaN,...,NaN,NaN,<NA>,0.00,0.01,0.0,NaN,NaN,NaN,NaN
2,00000nan,nan,BRAND,202612,2025-08-01,0.00,0.93,0.93,NaN,100.00,...,100.0,0.0,1,6.67,0.02,0.0,100.0,100.0,40.61,NaN
3,00000nan,nan,BRAND,202612,2025-08-08,8.32,1958.44,1747.60,NaN,89.23,...,100.0,0.0,1,15.79,0.08,0.0,100.0,100.0,40.96,NaN
4,00000nan,nan,BRAND,202612,2025-08-15,-8.47,1927.05,-1927.05,NaN,100.00,...,100.0,0.0,1,20.00,0.08,0.0,100.0,100.0,40.83,NaN


In [203]:
final_df['level'].value_counts()

level
BRAND                        6133
BRAND_MATERIAL_TYPE           400
BRAND_DOSAGE_FORM_PARENT      224
BRAND_MATERIAL_PLANT          223
BRAND_NODETYPE                170
BRAND_PLANT_TYPE              159
BRAND_PLANT                   150
BRAND_MATERIAL                118
BRAND_ENTERPRISE_CATEGORY     102
Name: count, dtype: int64

In [204]:
f = final_df[final_df["corp_brand_id"]!="00000nan"]

In [205]:
f.shape

(7650, 26)

In [206]:
final_df.shape

(7679, 26)

In [207]:
f.to_csv('Allbrands_agg_withDynamicLogic_v2.csv')

In [421]:
# brands_to_keep = [
#     "All Other Pharmaceut"
#      ,
#     "ABRAXANE"
#     ,
#     "REVLIMID"
# ]
# dates_to_keep = ["202612", "202712", "202812"]

# fil = final_agg_df[
#     final_agg_df["corporate_brand"].isin(brands_to_keep) &
#     final_agg_df["date"].isin(dates_to_keep)
# ]

In [425]:
# final_agg_df.to_csv('Allbrands_agg_file.csv')